In [1]:
# pip install windfreak
import json
import serial.tools.list_ports
from windfreak import SynthHD
import time
import pyvisa


In [2]:
rm = pyvisa.ResourceManager()
instruments = rm.list_resources()
yoko = None
for instrument in instruments:
    if 'YOKOGAWA' in instrument:
        # Initialize instrument
        yoko = rm.open_resource(instrument) 
        break
print(yoko.query('*IDN?').strip())

YOKOGAWA,GS210,91U322681,2.02


In [3]:
ports = serial.tools.list_ports.comports()
#ports
for port in ports:
    print(f"Port Name:   {port.device}")
    print(f"Description: {port.description}")
    print(f"Hardware ID: {port.hwid}")  # Shows USB Vendor ID, Product ID, and Location string
    print("-" * 30)
    if 'A3E5' in port.hwid:
        print(f"Windfreak is on COM port {port.device}")

Port Name:   COM4
Description: USB Serial Device (COM4)
Hardware ID: USB VID:PID=0483:A3E5 SER=2068338D5232
------------------------------
Windfreak is on COM port COM4
Port Name:   COM3
Description: Intel(R) Active Management Technology - SOL (COM3)
Hardware ID: PCI\VEN_8086&DEV_02E3&SUBSYS_099F1028&REV_00\3&11583659&1&B3
------------------------------


In [4]:
synth = SynthHD("COM4")
synth.init()
channel_a = synth[0]
channel_a.enable = True
channel_a.frequency = 13.0e9   # JPA pump frequency in Hz
channel_a.power = 3.0        # JPA pump power in dBm

channel_b = synth[1]
channel_b.enable = True
channel_b.frequency = 8.75e9   # TWPA pump frequency in Hz
channel_b.power = -25        #  TWPA pump power in dBm

In [6]:
channel_b.frequency

8750000000.0

In [7]:
channel_a.enable = True

In [8]:
amplifier_state = {}
yoko.write(":SOUR:FUNC VOLT")
amplifier_state['yoko_units'] = "volts"
amplifier_state['yoko_level'] = float(yoko.query(":SOUR:LEV?"))
amplifier_state['yoko_on'] = yoko.query(":OUTP?")

amplifier_state['jpa_pump_frequency'] = channel_a.frequency
amplifier_state['jpa_pump_power'] = channel_a.power
amplifier_state['jpa_pump_on'] = channel_a.enable
amplifier_state['twpa_pump_frequency'] = channel_b.frequency
amplifier_state['twpa_pump_power'] = channel_b.power
amplifier_state['twpa_pump_on'] = channel_b.enable

with open("amplifier_state.json", "w", encoding="utf-8") as f:
    json.dump(amplifier_state, f, indent=4)

In [15]:
with open("amplifier_state.json", "r") as f:
    amplifier_state = json.load(f)

yoko.write(":SOUR:FUNC VOLT")
yoko.write(":SOUR:LEV " + str(amplifier_state['yoko_level']))
if amplifier_state['yoko_on'] == True:
    yoko.write(":OUTP ON")
else:
    yoko.write(":OUTP OFF")

channel_a.enable = amplifier_state['jpa_pump_on']
channel_a.frequency = amplifier_state['jpa_pump_frequency']   # JPA pump frequency in Hz
channel_a.power = amplifier_state['jpa_pump_power']        # JPA pump power in dBm

channel_b.enable = amplifier_state['twpa_pump_on']
channel_b.frequency = amplifier_state['twpa_pump_frequency']   # TWPA pump frequency in Hz
channel_b.power = amplifier_state['twpa_pump_power']        #  TWPA pump power in dBm